In [ ]:
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy.stats import pearsonr, boxcox, yeojohnson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import pacf, adfuller, kpss
from sklearn.linear_model import LinearRegression

np.random.seed(0)

# Stationarity

**Stationarity** is a fundamental domain within time series analysis. Understanding and achieving stationarity is crucial because many powerful time series models (such as ARIMA) rely on the assumption that the data exhibits this property.

In essence, a time series is **stationary** if its statistical properties, specifically its mean, variance, and autocorrelation structure, remain constant over time. This means that if you were to take any segment of the time series, its statistical characteristics would be largely similar to any other segment of the same length.

The key characteristics of a stationary time series are:
- **Constant Mean**
- **Constant Variance**
- **Constant Autocorrelation Structure (No Seasonality or Trend)**

The first two properties are relatively straightforward: the series should not exhibit a long-term upward or downward drift (mean), and its fluctuations should remain consistent over time (variance). The third property, **constant autocorrelation structure**, is particularly interesting. It means that the relationship between the series and its past values (lags) stays consistent throughout time. If this structure changes, it typically signals the presence of evolving patterns, such as:
- **Trends**, where the strength or direction of movement shifts over time.
- **Seasonality**, where repeating cycles increase or decrease in magnitude.

In a stationary process, such effects are either absent or have been successfully removed.

Why is stationarity so important?
Non-stationary time series are notoriously difficult to model and forecast accurately. When the statistical properties of a series change over time, any model built on historical data may not accurately capture future behavior. By transforming a non-stationary series into a stationary one, we can:
1.  **Simplify Modeling:** Many statistical time series models are designed for stationary data, making them easier to apply and interpret.
2.  **Ensure Reliable Inferences:** Statistical tests and confidence intervals derived from models are often only valid if the underlying data is stationary.
3.  **Improve Forecast Accuracy:** Predictions from models built on stationary data tend to be more stable and reliable.

This notebook will delve into the concept of stationarity, explore how to identify its violations, and demonstrate various techniques to transform non-stationary time series into stationary ones.

## Isnt this just White Noise?

While the above conditions look very similar to white noise, there is a subtle difference. 
- Stationarity: mean is constant
- White noise: mean is 0

This is illustrated in the plot below.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), sharey=True)

n = 1000
x = np.arange(n)

white_noise = np.random.normal(0, 1, n)
axs[0].plot(x, white_noise)
axs[0].set_title("White noise $\\mu=0$")
axs[0].hlines(0, 0, n, color="orange", linestyle="dashed", label="Mean")

stationarity = np.random.normal(5, 1, n)
axs[1].plot(x, stationarity)
axs[1].set_title("Stationary $\\mu\\ne0$")
axs[1].hlines(5, 0, n, color="orange", linestyle="dashed", label="Mean")

fig.tight_layout()
plt.legend(bbox_to_anchor=(1.4, 1))

While white noise *is* a stationary series, not all stationary series *are* white noise.

## Violating Stationarity Assumptions

To further illustrate the concept of stationarity, let's examine three examples where one of the key assumptions is intentionally violated, while the others are held constant:
- A series with constant mean and seasonality, but changing variance
- A series with changing mean (trend), but constant variance and no seasonality
- A series with constant mean and variance, but seasonal structure

In [ ]:
# helper function
def plot_series_with_rolling_stats(x, y, window=50, title="Time Series", figsize=(8, 5)):
    """
    Plot a time series along with its rolling mean and rolling variance.
    """
    # Rolling statistics
    rolling_mean = sliding_window_view(y, window).mean(axis=1)
    rolling_var = sliding_window_view(y, window).var(axis=1)

    # Pad to match x axis
    pad = [np.nan] * (window - 1)
    mean_padded = np.concatenate([pad, rolling_mean])
    var_padded = np.concatenate([pad, rolling_var])

    # Plot
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    # Original series
    axes[0].plot(x, y, color="blue", alpha=0.7)
    axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_title(f"{title}")

    # Rolling mean
    axes[1].plot(x, y, color="blue", alpha=0.3)
    axes[1].plot(x, mean_padded, color="red")
    axes[1].set_title(f"Rolling Mean (window={window})")

    # Rolling variance
    axes[2].plot(x, y, color="blue", alpha=0.3)
    axes[2].plot(x, var_padded, color="green")
    axes[2].set_title(f"Rolling Variance (window={window})")
    axes[2].set_xlabel("Time")

    plt.tight_layout()
    plt.show()

In [ ]:
x = np.linspace(0, 100, 1000)
scales = np.linspace(3, 0.5, len(x))
y1 = np.random.normal(loc=0, scale=scales)
plot_series_with_rolling_stats(x, y1, title="1: Constant Mean, Changing Variance")

In [ ]:
# changing mean, constant variance
# these coeffs are purely arbitary, and took me way too long to find out
y2 = np.polyval([3e-04, -6e-02, 3.2e+00, -2e+01], x) + np.random.normal(0, 2, len(x))
plot_series_with_rolling_stats(x, y2, title="2: Changing Mean, Constant Variance")

In [ ]:
y3 = 1 * np.sin(2 * np.pi * x / 15) + np.random.normal(0, 0.5, len(x))
# as the period is 15, increase window size to show constant mean
plot_series_with_rolling_stats(x, y3, title="3: Seasonality", window=150)

## Detecting Trends and Seasonality

### Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF)

Autocorrelation is the measure of correlation between a time series, and a lagged version of itself. Given a time series $ y_t$, and some lag of $ k $, the autocorrelation of $ y_t $ can be defined as:
$$ Corr(y_t,\ y_{t-k}) $$
More formally, the autocorrelation coefficient at lag $k$ is often represented as:
$$\rho_k = \frac{Cov(y_t, y_{t-k})}{\sqrt{Var(y_t)Var(y_{t-k})}}$$
This tells us how well the past values of the series predict the present, based on linear relationships.

**It's important to note that the clear interpretations of ACF and PACF discussed below are most directly applicable to *stationary* time series.** 

This section aims to provide intuition behind the two main *flavours* of autocorrelation:
- Autocorrelation Function (ACF)
- Partial Autocorrelation Function (PACF)

Let's first set up an example time-series:
$$ y_t,\ y_{t-1},\ y_{t-2},\ ... $$
To provide a more concrete example, let's use months as the time:
$$ y_{Mar},\ y_{Feb},\ y_{Jan},\ ... $$
Now suppose this time-series represented stock prices, or temperature; it would be reasonable to assume that:
- *January's* value effects *February's*
- *February's* value effects *March's*
- And so on...

This is demonstrated below:

```mermaid
flowchart LR

A(Jan) --> |Effects| B(Feb)
B --> |Effects| C(Mar)
```

However, it is often that in real-world data past values influence each other recursively, that is:

```mermaid
flowchart LR

A --> |Effects| C
A(Jan) --> |Effects| B(Feb)
B --> |Effects| C(Mar)
```

Now let's suppose that we are interested in the correlation at lag $ k=2 $:
$$ Corr(y_{Mar},\ y_{Jan}) $$
We could simply calculate the Pearson's correlation between $ y_{Mar} $ and $ y_{Jan} $ , $ y_{Apr} $ and $ y_{Feb} $ , $ y_{May} $ and $ y_{Mar} $ and so on... However, this would capture both of the following relationships:
- Direct influence of $ Jan \rightarrow Mar $
- Indirect influence of $ Jan \rightarrow Feb \rightarrow Mar $

What if we were *only* interested in the direct influnce *Jan* has on *Mar*. This brings us to the distinction between ACF and PACF:
- ACF includes *all* direct and indirect relationships between the original series, and it's lagged version
- PACF allows us to investigate the relationships in isolation, effectively removing the influence of intermediate lags.

#### ACF

Much like we have done previously, let's generate a noisy sine signal to use as an example. Note: we create the signal with 100 data points, and a period of 20, as such we should expect to see 5 cycles.

In [ ]:
n = 100
t = np.arange(n)

period = 20
y = 1 * np.sin(2 * np.pi * t / period) + np.random.normal(0, 0.5, n)

plt.figure(figsize=(8, 5))
plt.plot(t, y, label='Time Series')
plt.title('Synthetic Time Series')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.show()

As expected, there are 5 clear peaks in the data. Let's apply a lag to the above data.

In [ ]:
fig, axes = plt.subplots(figsize=(8, 5))

# plot original
plt.plot(t, y, label='Original')

# apply lag
k = 5
y_lag = y[k:]
y_lag = np.pad(y_lag, (0, k))
plt.plot(t, y_lag, label='Lagged', alpha=0.5)

axes.set(xlabel="Time", ylabel="Value", title=f"Lagged series ({k=})")
plt.legend()
plt.tight_layout()

To illustrate the relationship (if any) of the above data, we can plot both series against each other.

In [ ]:
fig, axes = plt.subplots(figsize=(8, 5))
plt.scatter(x=y_lag, y=y, label="$y_t\\ vs\\ y_{t-k}$")

# plot linear slope
m, b = np.polyfit(y_lag, y, deg=1)
plt.axline(xy1=(0, b), slope=m, label=f'$y = {m:.2f}x {b:+.2f}$', linestyle="--", color="red")

# find pearsons
pearsons_corr, _ = pearsonr(y, y_lag)

axes.set(title=f"Original vs Lagged ({k=}, pearson={pearsons_corr:0.2f})", xlabel=f"y (lag={k})", ylabel="y")

plt.legend()
plt.tight_layout()

For a *k* of 5, there exists no strong relationship between the two series. Let's explore different values of *k*, using the same process. Each subplot has:
- A scatter plot of $ y_t $ vs $ y_{t-k} $
- Best fit regression line
- The Pearson correlation coefficient

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 5), sharey=True)

for ax_i, k in enumerate((10, 15, 20)):
    y_lag = y[k:]
    y_lag = np.pad(y_lag, (0, k), mode="constant", constant_values=0)

    m, b = np.polyfit(y_lag, y, deg=1)

    pearsons_corr, _ = pearsonr(y, y_lag)
    
    axes[ax_i].scatter(x=y_lag, y=y)
    axes[ax_i].axline(xy1=(0, b), slope=m, label=rf"$y = {m:.2f}x \ {b:+.2f}$", linestyle="--", color="red")
    axes[ax_i].set(title=f"{k=}, pearsons={pearsons_corr:0.2f}")
    axes[ax_i].legend()

fig.supylabel("y")
fig.supxlabel("y_lag")
plt.tight_layout()

Observations:
- At $ k=10 $, the lag is half the period. This results in a strong negative correlation, as the peaks will be aligned with the troughs
- At $ k=15 $, the lag does not align with any harmonic of the orignal sine wave, and hence there is a near-zero correlation, with a near-flat regression line
- At $ k=20 $, the lag matches the period. This results in a strong positive correlation, as peaks align with peaks

#### Autocorrelation Plot

We could continue plotting the correlations of $ y_{t-k} $ on $ y_t $ for further lags of *k*, however, an *autocorrelation plot* is much easier to digest and interpret. `statsmodels`' `plot_acf` provides this functionality out-of-the-box. Before we create an autocorrelation plot for the example data we have been working with, let's review how to interpret one.

**Interpretation**
- An autocorrelation plot plots the correlation coefficient for varying lag values
- A bar outside of the confidence interval suggets that the observed correlation is unlikely due to random chance
- A bar inside the confidence interval are likely due to random noise and are not statistically significant

Alongside simply indicating which lag values produce significant (or not) correlations, the general *structure* of an autocorrelation plot can provide us with insights.

**Key structures**
- A slow decay to zero suggests that a series is non-stationary
- A rapid decay suggests that a series is stationary
- Large bars at regular intervals suggest periodicity and seasonality
- No significant bars is evidence of a random signal

Let's now perform **ACF**, up to $ k=50 $ (50 lags), with the above data.

In [ ]:
plot_acf(y, lags=50)
plt.title('Autocorrelation')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation Coefficient')

Observations:
- At $ k=0 $, the correlation will always be 1, as this is the correlation of the series with itself
- As expected, there are strong peaks every 20 steps

#### PACF

Let's quickly recap at a highlevel PACF and how it differs from ACF above, before introducing it more formally:
- ACF at lag *k* tells you:
    - "How correlated is the current value with the value *k* steps back, including all indirect paths?"
- PACF at lag *k* tells you:
    - "How correlated is the current value with the value *k* steps back, after accounting for the effects of all in-between lags?"
 
To demonstrate the PACF clearly, we’ll generate a new time series where each value is explicitly defined by its previous values. This is known as an autoregressive (AR) process. For example, an AR(1) process is where each value only depends on its immediate predecessor. Later notebooks will explore AR processes further. Unlike the earlier sine wave, which exhibits seasonality, an AR process is ideal for illustrating PACF because:
- The dependencies between time steps are controlled and known, making interpretation straightforward.
- PACF reveals the direct influence of past values

To continue with our original example of investigating the effect a value in *Jan* has on *Mar*, let's generate an AR(3) model, plus a bit of noise:
$$ y_t = 0.75y_{t-1} - 0.5y_{t-2} + 0.6y_{t-3} + \epsilon_t $$
Again, we'll revisit this model structure in more detail later. The above has been plotted below.

In [ ]:
def generate_ar_process(coeffs, n=100, noise_std=1):
    """Generate an AR(p) time series of length n with given coefficients."""
    p = len(coeffs)
    y = np.zeros(n)
    noise = np.random.normal(0, noise_std, n)
    for t in range(p, n):
        y[t] = noise[t] + sum(coeffs[i] * y[t - i - 1] for i in range(p))
        
    return y

In [ ]:
y = generate_ar_process([0.75, -0.5, 0.6])

plt.plot(y)
plt.title("$ y_t = 0.75y_{t-1} - 0.5y_{t-2} + 0.6y_{t-3} + \\epsilon_t $")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

We will now re-iterate a point that was touched upon in [this section](#Autocorrelation-Function-(ACF)-and-Partial-Autocorrelation-Function-(PACF)). It was stated that, for example, a strong correlation between March and January might just reflect the fact that January affects February, and February affects March, i.e. an indirect influence, not that January directly influences March.

PACF aims to uncover:
- What is the direct effect of a past value (January) on the present (March), after accounting for everything in between?

To compute this, we fit a regression with all intermediate lags up to some *k*, and then for each, extract the coefficient on lag *k*. This coefficient, commonly denoted as $ \phi_{k,k} $, tells us the pure contribution of that lag. This is shown below.

For a given *k*, we fit the following linear regression model:
$$ y_t = \beta_1y_{t-1} + \beta_2y_{t-2} + \ldots + \beta_ky_{t-k} + \epsilon_t$$
Where:
- $ y_t $ is the target value (current value)
- $ y_{t-k} $ is the lagged predictor for a lag of *k*
- $ \beta_k $ is the coefficient for lag *k*
- $ \epsilon_t $ is residual white noise
The partial autocorrelation coefficient for lag *k* is given as:
$$ \phi_{k,k} = \beta_k $$

Much like we did for ACF, let's work through an example, for lags up to k=3, starting with k=1.

In [ ]:
k=1
# we plus one to include the current value, i.e. y_target
windows = sliding_window_view(y, k+1)
X, y_target = windows[:, :-1], windows[:, -1]

# fit a linear model
lr = LinearRegression().fit(X, y_target)

Before we examine the above coefficients to find $ \phi_{1,1} $ there is a nuance to the implementation above method that must be accounted for. 
- In the regression formula above, the coefficients $ \beta_1, \beta_2, \ldots, \beta_k $ correspond to the lags $ y_{t-1} + y_{t-2} + \ldots + y_{t-k} $ in a **right-to-left** order

However, when using `Numpy`'s `sliding_window_view` function, the lagged features are typically arranged **left-to-right**, with the oldest lag first. I.e:
```python
X = [y_{t-k}, ..., y_{t-2}, y_{t-1}]
```
Therefore, the coefficient for lag *k* $ \beta_k $ will always appear at **index 0** in the fitted linear model's coefficients. 

Now examining the coefficients for lag 1, we find $ \phi_{1,1} $ to be:

In [ ]:
print(lr.coef_[0])

Repeating for k=2,3.

In [ ]:
for k in (1, 2, 3):
    windows = sliding_window_view(y, k+1)
    X, y_target = windows[:, :-1], windows[:, -1]
    
    # fit a linear model
    lr = LinearRegression().fit(X, y_target)

    # extract phi_{k,k}: the coefficient on y_{t-k}
    phi_kk = round(lr.coef_[0], 2)

    print(f"phi_{k},{k} = {phi_kk}")

From the manual regression approach, we obtain:

- $ \phi_{1,1} = 0.5 $  
- $ \phi_{2,2} = -0.24 $  
- $ \phi_{3,3} = 0.55 $

These values represent the **partial autocorrelation coefficients**. Summarising:
- The relatively strong $ \phi_{1,1} $ and moderate $ \phi_{3,3} $ values indicate notable direct effects from lags 1 and 3. This confirms a significant direct influence from lag 3, consistent with the AR(3) model that generated the data.
- The comparatively smaller $ \phi_{2,2} $ suggests that lag 2 has a weaker direct influence on the current value, even if it may contribute indirectly via other lags.

Let's use `statsmodels`' `plot_pacf` function to confirm the above method, and plot further lags.

In [ ]:
plot_pacf(y, lags=50, zero=False)
plt.title('Partial Autocorrelation Plot')
plt.xlabel('Lag')
plt.ylabel('Partial Autocorrelation Coefficient')

Much like an ACF plot:
- Large bars represent statistically significant influence
- Bars within the confidence interval show no statistically significant effect

**Observations**
- Firsty, we can visually confirm that the plot agrees with the first three *phi* values generated via the manual regression method
- $ \phi_{2,2} $ is statistically non-significant
- There is a sharp drop off after lag 3. This is a typical structure of partial autocorrelation plots, where for an AR(*p*) process, the partial autocorrelation values sharply drop off at lag *p*

This sharp cutoff in the partial autocorrelation function is particularly useful in practice because it helps identify the order *p* of an AR process when modeling real-world time series.

### Summary

The autocorrelation function (ACF) and partial autocorrelation function (PACF) are essential tools to understand the internal dependencies of a time series. While the ACF captures both direct and indirect correlations between values at different lags, the PACF isolates the direct effect of each lag by controlling for intermediate lags.

Interpreting the shape and cutoff patterns in these plots provides valuable clues about the underlying process, such as the order of an autoregressive model, and helps inform model selection.

## Statistial Tests for Stationarity

This section introduces two of the most commonly used statistical tests for assessing stationarity:
1. Augmented Dickey-Fuller (ADF)
2. Kwiatkowski-Phillips-Schmidt-Shin (KPSS)

These are often used together, as one supplements the other, which allows us to form a more robust understanding of whether a time series is stationary. Before moving to these, we must first discuss a topic that hasn't been covered in this notebook, **Unit Roots**. 

### Unit Roots

To begin understanding unit roots, we first need to define **shock**:
- **Shock** can be defined as a sudden event that causes a significant, extreme change in the data
- We model any shock with the inclusion of the white noise term $ \epsilon_t $ in a time series model

Below is a simulated shock event at time $t=30 $, and three different scenarios that show how a time series might respond:

1. Permanent Effect: the shock has a lasting, unbounded impact  
2. Temporary Shock with Reversion: the series returns to its original baseline  
3. Temporary Shock with Mean Shift: the series stabilises around a new mean

Each behavior corresponds to a different kind of process: non-stationary with a **unit root**, stationary, or **trend-stationary / structural break**.

In [ ]:
base_mean = 0
new_mean = 7.5
n_pre = 30
shock_value = 15
n_total = 100
transition_len = 30

# Base signal before shock
y_pre = np.random.normal(base_mean, 1, n_pre + 1)
y_pre[-1] += shock_value  # apply shock at final pre-shock point

# Permanent effect (unit root / explosive behavior)
y_explosive = np.cumsum(np.random.normal(0.25, 0.25, n_total - len(y_pre)))

# Transition x values
x_transition = np.arange(n_pre, n_pre + transition_len)

# 1. Transition to new mean
linear_new = np.linspace(y_pre[-1], new_mean, transition_len)
noise_new = np.random.normal(0, 0.5, transition_len)
y_transition_new = linear_new + noise_new
y_new_mean = np.random.normal(new_mean, 0.5, n_total - len(y_pre) - transition_len)

# 2. Transition back to base mean
linear_base = np.linspace(y_pre[-1], base_mean, transition_len)
noise_base = np.random.normal(0, 0.5, transition_len)
y_transition_base = linear_base + noise_base
y_base_mean = np.random.normal(base_mean, 0.5, n_total - len(y_pre) - transition_len)

# Plotting
fig, axes = plt.subplots(figsize=(8, 5))

plt.plot(np.concatenate((y_pre, y_explosive + shock_value)), color="tab:red", label="Permanent Effect")
plt.plot(np.concatenate((y_pre, y_transition_new, y_new_mean)), color="tab:green", label="Temporary Shock $ \\rightarrow $ New Mean")
plt.plot(np.concatenate((y_pre, y_transition_base, y_base_mean)), color="tab:orange", label="Temporary Shock $ \\rightarrow $  Baseline")

plt.plot(y_pre, color="tab:blue", label="Base Signal")

# Labels and legend
plt.title("Responses to a Shock in Different Time Series Processes")
plt.xlabel("Time")
plt.ylabel("Value")
plt.axvline(x=n_pre, color='black', linestyle='--', alpha=0.6, label="Shock Event")
plt.legend(loc='upper left', bbox_to_anchor=(0, 1), fontsize="9")
plt.tight_layout()

#### What is a Unit Root?

In the plot above, we saw how a time series might respond after a shock, in some cases the shock has a lasting effect, while in others the series returns to its previous behavior. This is sometimes referred to as "memory".

This difference in can be explained by the presence or absence of a **unit root**:
- A unit root indicates that a time series is **non-stationary**, meaning the effect of a shock persists indefinitely.

Let’s examine a simple autoregressive process of order 1, AR(1):
$$ y_t = \phi y_{t-1} + \epsilon_t $$

The value of $ \phi $ determines the behavior of the series:
- $ |\phi| < 1 $: shocks decay over time, the series reverts to a mean, the series is stationary
- $ \phi = 1 $: shocks accumulate and *randomly walk*, the series is non-stationary (unit root)
- $ |\phi| > 1 $: shocks grow over time; the series increases explosively, the series is non-stationary

The above can be generalized to an AR(*p*) process with multiple lag terms, but the details are beyond the scope of this notebook and often less useful from a practical perspective.

### Augmented Dickey-Fuller (ADF)

The Dickey-Fuller test is a statistical test used to check for a unit root in a time series, i.e., to test if the series is non-stationary. The Dickey-Fuller test assumes an autoregressing process of order 1, AR(1), as such we have seen before:
$$ y_t = \phi y_{t-1} + \epsilon_t $$
Taking the first difference, i.e. subtracting $ y_{t-1} $: 
$$ y_t - y_{t-1} = \phi y_{t-1} + \epsilon_t - y_{t-1}$$
Rearranging:
$$ \Delta y_t = (\phi - 1) y_{t-1} + \epsilon_t $$
Substituting $ \delta $ for $ (\phi - 1) $:
$$ \Delta y_t = \delta y_{t-1} + \epsilon_t $$
We can now formally define the Dickey-Fuller hypotheses:
- $ H_0: \delta = 0 $ (There exists a unit root, the series is non-stationary)
- $ H_a: \delta < 0 $ (No unit root exists, and such the series is stationary)

However, what is our series is more complex than AR(1)? This is where the **Augmented** Dickey-Fuller (ADF) test comes in. The ADF generalises the above, by adding a summation over all *p* lagged differences, for a general AR(*p*) process. The ADF equation is given below, to illustate the inclusion of all lagged differences:
$$ \Delta y_t = \delta y_{t-1} + \sum_{i=1}^{p} \beta_i \Delta y_{t-i} + \epsilon_t$$
It can be seen that the lagged terms $ \Delta y_{t-i} $ account for all possible lagged differences, up to *p*. The crucial note is that between the **DF** and the **ADF**, the hypotheses **do not** change:
- $ H_0: \delta = 0 $ (There exists a unit root, the series is non-stationary)
- $ H_a: \delta < 0 $ (No unit root exists, and such the series is stationary)

#### Examples

To illustrate the ADF, we will now create four time series:
1. A stationary AR(1) process  
2. A non-stationary AR(1) process (with a unit root)  
3. A stationary AR(3) process  
4. A non-stationary AR(3) process (with near-unit root behavior)

For each series, we will visualise the signal, rolling mean, and rolling variance (as we have done in [Violating Stationarity Assumptions](#violating-stationarity-assumptions)), and apply the ADF test to examine stationarity.

- If the p-value > 0.05, we **fail to reject the null hypothesis** (unit root present), which means the series is **non-stationary**.

In [ ]:
x = np.linspace(0, 100, 100)

In [ ]:
def simple_adf(series):
    result = adfuller(series)
    adf_stat, p_value = result[0], result[1]
    return f"ADF: {adf_stat:.4f}, p-value: {p_value:.4f}"

**AR(1): Stationary**

In [ ]:
ar1_stationary = generate_ar_process([0.7], 100)

In [ ]:
result = simple_adf(ar1_stationary)
print(result)

In [ ]:
plot_series_with_rolling_stats(x, ar1_stationary, window=50, title=f"AR(1): Stationary\n{result}")

**AR(1): Non-stationary**

In [ ]:
ar1_non_stationary = generate_ar_process([1.0], 100)

In [ ]:
result = simple_adf(ar1_non_stationary)
print(result)

In [ ]:
plot_series_with_rolling_stats(x, ar1_non_stationary, window=50, title=f"AR(1): Non-stationary\n{result}")

**AR(3): Stationary**

In [ ]:
ar3_stationary = generate_ar_process([0.5, -0.3, 0.2], 100)

In [ ]:
result = simple_adf(ar3_stationary)
print(result)

In [ ]:
plot_series_with_rolling_stats(x, ar3_stationary, window=50, title=f"AR(3): Stationary\n{result}")

**AR(3): Non-stationary**

In [ ]:
ar3_non_stationary = generate_ar_process([1.3, -0.4, 0.1], 100)

In [ ]:
result = simple_adf(ar3_non_stationary)
print(result)

In [ ]:
plot_series_with_rolling_stats(x, ar3_non_stationary, window=50, title=f"AR(3): Non-stationary\n{result}")

### Kwiatkowski-Phillips-Schmidt-Shin (KPSS)

The KPSS test is based around two fundamental concepts:
1. Linear regression
2. Residual analysis

And it can take on two forms, each of which test a different type of stationarity:
1. Constant stationarity: The series is stationary around some constant
  $$
  y_t = \mu + \epsilon_t
  $$
2. Trend stationarity: The series is stationary around a trend.
  $$
  y_t = \mu + \beta t + \epsilon_t
  $$

However, behind the scenes, the KPSS test assumes the time series can be broken into three components:
$$
y_t = \mu + r_t + \epsilon_t
$$
Where:
- $ \mu $ is a constant (or a deterministic trend if testing for trend stationarity),
- $ r_t $ is a random walk component,
- $ \epsilon_t $ is an error term.

In essence, the KPSS fits a linear regression, and examines the residuals to see if they are white noise (i.e. the series is stationary) or if they have structure (suggestive of non-stationarity). It is important to note that KPSS assumes the opposite hypotheses compared to ADF, that is:
- $ H_0: $ The series is stationary (level or trend stationary, depending on the form of the test)
- $ H_a: $ The series is non-stationary

#### Examples - Constant Stationarity

To illustrate the KPSS test, we will use the 4 series generated above in the ADF Examples, where we test for constant stationarity.

- If the p-value > 0.05, we **fail to reject the null hypothesis** (series is stationary), which means the series is **stationary**.

In [ ]:
def simple_kpss(series, regression='c'):
    """
    Suppresses InterpolationWarning which occurs when the test statistic
    falls outside the range of available p-values in the lookup table.
    This means the actual p-value is either smaller or larger than reported,
    indicating very strong evidence for stationarity or non-stationarity.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        stat, p_value, _, _ = kpss(series, regression=regression, nlags="auto")
    return f"KPSS: {stat:.4f}, p-value: {p_value:.4f}"

**AR(1): Stationary**

In [ ]:
result = simple_kpss(ar1_stationary)
print(result)

**AR(1): Non-stationary**

In [ ]:
result = simple_kpss(ar1_non_stationary)
print(result)

**AR(3): Stationary**

In [ ]:
result = simple_kpss(ar3_stationary)
print(result)

**AR(3): Non-stationary**

In [ ]:
result = simple_kpss(ar3_non_stationary)
print(result)

#### Examples - Trend Stationarity

As previously stated, the KPSS test also allows for the testing of trend stationarity by setting `regression='ct'`. This checks whether a time series is stationary around a deterministic trend.

Below, we demonstrate two further examples using this setting, one linear and one non-linear trend, both with added random noise.


**Linear**

In [ ]:
n = 100
t = np.arange(n)
linear_trend = 0.1 * t
noise = np.random.normal(0, 1, n)
series_linear = linear_trend + noise

# Plot
plt.figure(figsize=(8, 4))
plt.plot(series_linear, label="Linear Trend + Noise")
plt.title("Trend Stationary Series (Linear Trend + Noise)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()

# KPSS test
print("Linear trend + noise:", simple_kpss(series_linear, regression='ct'))

**Non-Linear**

In [ ]:
# Generate non-linear series
nonlinear_trend = 0.001 * (t ** 2)
series_nonlinear = nonlinear_trend + noise

# Plot
plt.figure(figsize=(8, 4))
plt.plot(series_nonlinear, label="Non-linear Trend + Noise", color="tab:orange")
plt.title("Non-Stationary Series (Non-linear Trend + Noise)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()

# KPSS test
print("Non-linear trend + noise:", simple_kpss(series_nonlinear, regression='ct'))

### ADF / KPSS: Summary

The ADF and KPSS tests are complementary tools to assess stationarity in time series data.
- ADF: assumes the presence of a unit root (i.e, non-stationarity) as the null hypothesis.
- KPSS: assumes stationarity as the null hypothesis.

**Note:**  
- ADF: a p-value > 0.05 means we fail to reject the null (presence of a unit root), series is non-stationary.  
- KPSS: a p-value > 0.05 means we fail to reject the null (stationarity), series is stationary.

By using both tests together, we can gain a clearer understanding of whether a series is stationary or non-stationary.
| ADF Test         | KPSS Test        | Interpretation                          |
|------------------|------------------|------------------------------------------|
| Fail to reject   | Reject           | Likely non-stationary                    |
| Reject           | Fail to reject   | Likely stationary                        |
| Fail to reject   | Fail to reject   | Inconclusive                             |
| Reject           | Reject           | Possible structural break or instability |


Interpreting both results together helps avoid false conclusions and better characterises the time series.

## Converting to Stationary

In this section, we will cover common methods for converting non-stationary time series into stationary time series.

In [ ]:
def stationarity_tests(series, regression='c'):
    # Suppress KPSS warnings about p-value interpolation
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        kpss_stat, kpss_p, _, _ = kpss(series, regression=regression, nlags="auto")

    adf_stat, adf_p, _, _, _, _ = adfuller(series)

    adf_result = f"ADF: stat={adf_stat:.4f}, p={adf_p:.4f}"
    kpss_result = f"KPSS: stat={kpss_stat:.4f}, p={kpss_p:.4f}"

    return f"{adf_result} | {kpss_result}"

### Differencing

Differencing is a method where we subtract the previous observation(s) from the current observation to remove trends or make a time series stationary. By differencing, we eliminate persistent effects like trends or unit roots that cause non-stationarity. This operation often stabilises the mean of the series.

The first difference of a series $ y_t $ is given by:
$$ y'_t = y_t - y_{t-1} $$
The second difference is simply the first difference of the first difference:
$$ y''_t = y'_t - y'_{t-1} $$
Differencing can be applied multiple times if needed (e.g., for polynomial trends).

**When to use it?**  
- Use differencing when the series shows a trend or contains a unit root, indicated by tests like ADF.  
- Over-differencing can introduce unnecessary complexity and remove meaningful information.  
- If the series is seasonal, consider removing seasonality before differencing.

Below, we simulate a non-stationary AR(1) process, apply first differencing, and visualise the results.

In [ ]:
# AR(1)
y = generate_ar_process([1], 100)

# First difference
y_diff = np.diff(y)

# Plot original and differenced series
fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axs[0].plot(y, label="Original AR(1)", color="tab:blue")
axs[0].set_title(f"Non-Stationary Series with Unit Root\n{stationarity_tests(y)}")
axs[0].legend()

axs[1].plot(y_diff, label="First Difference", color="tab:orange")
axs[1].set_title(f"Differenced Series (Stationary)\n{stationarity_tests(y_diff)}")
axs[1].legend()

plt.tight_layout()

### Detrending

Detrending involves removing a deterministic trend component (such as linear or polynomial) from a time series, by fitting a regression model and subtracting the fitted trend. Detrending helps to stabilise the mean and make the series stationary.

For a linear trend, we model the series as:
$$ y_t = \beta_0 + \beta_1 t + u_t $$
where $ \beta_0 + \beta_1 t $ is the deterministic trend and $ u_t $ is the detrended residual series.
Detrending involves estimating $ \beta_0 $ and $ \beta_1 $, then subtracting the trend:
$$ y_t^{det} = y_t - (\hat{\beta}_0 + \hat{\beta}_1 t) $$

**When to use it?**  
- when the series shows a deterministic trend rather than a stochastic trend (unit root).  
- If differencing removes too much information or introduces unnecessary noise.  
- If there is seasonality or other components, remove those separately.


In [ ]:
n = 100
t = np.arange(n)

# Generate series with linear trend + noise
trend = 0.1 * t
noise = np.random.normal(0, 1, n)
y = trend + noise

# Fit linear regression for detrending
model = LinearRegression()
model.fit(t.reshape(-1, 1), y)
trend_pred = model.predict(t.reshape(-1, 1))

# Detrended series
y_detrended = y - trend_pred

# Plot original, trend, and detrended series
fig, axs = plt.subplots(3, 1, figsize=(8, 5), sharex=True)

axs[0].plot(y, label="Original Series")
axs[0].set_title(f"Original Series with Linear Trend\n{stationarity_tests(y)}")
axs[0].legend()

axs[1].plot(trend_pred, label="Fitted Trend", color='tab:green')
axs[1].plot(y, label="Original Series", alpha=0.7)
axs[1].set_title("Estimated Linear Trend")
axs[1].legend()

axs[2].plot(y_detrended, label="Detrended Series", color='tab:orange')
axs[2].set_title(f"Detrended Series (Stationary)\n{stationarity_tests(y_detrended)}")
axs[2].legend()

plt.tight_layout()

### Deseasonalising

#### Seasonal Differencing

One simple way to remove seasonality is to subtract the value from the same period in the previous cycle. 

In [ ]:
# arbitary 5 years of monthly data
n = 60
period = 12
t = np.arange(n)

# seasonal + noise
seasonal = 5 * np.sin(2 * np.pi * t / period)
noise = np.random.normal(0, 1, n)
y = seasonal + noise
series = pd.Series(y)

# here we diff() with the period of the data
y_diff = series.diff(periods=period)
y_diff = y_diff.dropna()

fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axs[0].plot(series, label="Original Series")
axs[0].set_title(f"Original Series with Seasonality\n{stationarity_tests(series)}")
axs[0].legend()

axs[1].plot(y_diff, color="tab:orange", label="Seasonally Differenced Series")
axs[1].set_title(f"After Seasonal Differencing\n{stationarity_tests(y_diff)}")
axs[1].legend()
plt.tight_layout()

#### Manual Seasonal Mean Removal  

In [ ]:
# create datetime index for grouping above data
date_range = pd.date_range(start="2000-01-01", periods=n, freq="ME")
series = pd.Series(y, index=date_range)

# subtract monthly means
monthly_means = series.groupby(series.index.month).transform("mean")
deseasonalised = series - monthly_means

fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axs[0].plot(series, label="Original Series")
axs[0].set_title(f"Original Series with Seasonality\n{stationarity_tests(series)}")
axs[0].legend()

axs[1].plot(deseasonalised, color="tab:green", label="Monthly Means Removed")
axs[1].set_title(f"Deseasonalised Series\n{stationarity_tests(deseasonalised)}")
axs[1].legend()

plt.tight_layout()

## Log Transform, and Variants

Log transformations work by applying the logarithm function to each value in a time series. For a time series $X$, where:
$$ X = [X_t, X_{t-1}, X_{t-2}, \ldots, X_{t-n}] $$
The log-transformed series is then given by:
$$ X^{\log} = [\log(X_t), \log(X_{t-1}), \log(X_{t-2}), \ldots, \log(X_{t-n})] $$

Log transforms have two primary uses within time series analysis:
1. Convert series that exhibit exponential trends to a linear trend.
2. Address *heteroscedasticity*, where the variance is non-constant and instead proportional to the series magnitude.

These two distinct use cases have been demonstrated below. A key limitation of log transforms is that the data must be **non-negative** for the logarithm to be defined.

**Note**: The following examples were generated using LLM's and then modified. This was due to time constraints and other commitments.

In [ ]:
n_samples = 100
t = np.arange(n_samples)
epsilon = 1e-6

# --- Example 1: Linearising Exponential Trend (with additive noise) ---
A_exp, B_exp = 5, 0.05
base_offset_exp = 20
std_additive_noise = 2

series_exp_original = A_exp * np.exp(B_exp * t) + base_offset_exp + np.random.normal(0, std_additive_noise, n_samples)
series_exp_original = np.clip(series_exp_original, epsilon, None)
series_exp_transformed = np.log(series_exp_original)

# --- Example 2: Stabilising Variance (multiplicative noise on a linear trend) ---
C_linear, D_linear = 0.5, 30
std_multiplicative_noise_factor = 0.5

base_linear_trend = C_linear * t + D_linear
noise_multiplicative = np.random.normal(0, std_multiplicative_noise_factor, n_samples)
series_var_original = base_linear_trend * (1 + noise_multiplicative)

series_var_original = np.clip(series_var_original, epsilon, None)
series_var_transformed = np.log(series_var_original)


# --- Plotting ---
fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex='col') # Updated subplots call
fig.suptitle('Log Transformation\nLinearising Trends & Stabilising Variance', fontsize=16)

# Row 1: Exponential Trend Example
axes[0, 0].plot(t, series_exp_original, lw=1.5, color='tab:blue')
axes[0, 0].set_title("1a. Original (Exponential Trend)", fontsize=11)
axes[0, 0].set_ylabel("Value")
axes[0, 0].grid(True, linestyle='--', alpha=0.6)
axes[0, 0].set_ylim(bottom=0)

axes[0, 1].plot(t, series_exp_transformed, lw=1.5, color='tab:red')
axes[0, 1].set_title("1b. Log Transformed (Linear Trend)", fontsize=11)
axes[0, 1].set_ylabel("Log(Value)")
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

# Row 2: Variance Stabilization Example
axes[1, 0].plot(t, series_var_original, lw=1.5, color='tab:orange')
axes[1, 0].set_title("2a. Original (Increasing Variance)", fontsize=11)
axes[1, 0].set_ylabel("Value")
axes[1, 0].set_xlabel("Time")
axes[1, 0].grid(True, linestyle='--', alpha=0.6)
axes[1, 0].set_ylim(bottom=0)

axes[1, 1].plot(t, series_var_transformed, lw=1.5, color='tab:green')
axes[1, 1].set_title("2b. Log Transformed (Stabilised Variance)", fontsize=11)
axes[1, 1].set_ylabel("Log(Value)")
axes[1, 1].set_xlabel("Time")
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

#### Box-Cox

The Box-Cox transform is a generalised version of the log transform. It introduces a power parameter $ \lambda $ that is optimised to maximise the likelihood of the transformed data being normally distributed, or, specifically for time series, to stabilise the variance over time. The transformation is given as:
$$ Y^{\lambda} = \begin{cases}
\frac{Y^{\lambda} - 1}{\lambda} & \text{if } \lambda \neq 0\\
\ln(Y) & \text{if } \lambda = 0 \\
\end{cases} $$
Where:
- $ Y $ is the original series
- Lambda $ \lambda $ is the power parameter
- $ Y^{\lambda} $ is the transformed series

A crucial point for the Box-Cox transformation, similar to the log transform, is that it is only applicable for strictly positive values ($Y > 0$).

To best illustrate the Box-Cox transformation in a time series context, we will focus on its ability to **stabilise variance over time**. While the Box-Cox transform can also help in achieving a more normal distribution, its key benefit in time series analysis is often making the variance of residuals constant (homoscedasticity), which is a crucial assumption for many forecasting models like ARIMA.

Below, we generate a synthetic time series that exhibits a clear linear trend but with **multiplicative noise**. This causes the magnitude of the fluctuations to visibly increase as the series values grow (heteroscedasticity). The Box-Cox transformation is then applied, and a statistical method is used to automatically determine the optimal $\lambda$ value that best stabilises this variance.

In [ ]:
n_samples = 100
t = np.arange(n_samples)
epsilon = 1e-6 # Small value to ensure positivity before transform

# --- Generate Original Series with Increasing Variance (Multiplicative Noise) ---
# A linear trend as a base, with amplitude increasing over time
base_trend = 0.8 * t + 50

# Introduce significant multiplicative noise to clearly show heteroscedasticity
# The noise factor determines the percentage variability around the base trend
multiplicative_noise_factor = 0.4 # e.g., 40% variability

# Series = Base Trend * (1 + random noise scaled by factor)
original_series_hetero = base_trend * (1 + np.random.normal(0, multiplicative_noise_factor, n_samples))

original_series_hetero = np.clip(original_series_hetero, epsilon, None)

# --- Apply Box-Cox Transformation ---
transformed_series_boxcox, optimal_lambda = boxcox(original_series_hetero)

# --- Plotting ---
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
fig.suptitle('Box-Cox Transformation: Stabilizing Variance in Time Series', fontsize=16)

# Plot Original Series
axes[0].plot(t, original_series_hetero, lw=1.5, color='tab:blue')
axes[0].set_title("Original Series (Linear Trend, Increasing Variance)", fontsize=12)
axes[0].set_ylabel("Value")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].set_ylim(bottom=0) # Ensure y-axis starts at 0 or slightly above

# Plot Box-Cox Transformed Series
axes[1].plot(t, transformed_series_boxcox, lw=1.5, color='tab:orange')
# Display the optimal lambda in the title
axes[1].set_title(f"Box-Cox Transformed Series (Optimal $\\lambda$ = {optimal_lambda:.4f})", fontsize=12)
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Transformed Value")
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

By observing the scale (y-axis) of the graph, we can see that the variance has been stabilised. However, it still shares a key limitation with the log transform: the data must be strictly positive. This restricts its use in many real-world scenarios where zero or negative values are present.

The Yeo-Johnson transformation addresses this limitation by extending the Box-Cox approach to work with zero and negative values.

#### Yeo-Johnson

The Yeo-Johnson transformation is an extension of the Box-Cox transformation that overcomes the limitation of requiring strictly positive data. It can handle zero and negative values. The transformation is defined piecewise:

For $Y \ge 0$:
$$ Y^{(\lambda)} = \begin{cases} \frac{(Y+1)^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(Y+1) & \text{if } \lambda = 0 \end{cases} $$
For $Y < 0$:
$$ Y^{(\lambda)} = \begin{cases} -\frac{(-Y+1)^{2-\lambda} - 1}{2-\lambda} & \text{if } \lambda \neq 2 \\ -\ln(-Y+1) & \text{if } \lambda = 2 \end{cases} $$

Again, like we saw in Box-Cox transform, the parameter $ \lambda $ is optimised. 

Below, we apply the Yeo-Johnson to an exponential series, that has been *y* shifted to create negative values.

In [ ]:
n = 100
x = np.arange(n)

# Generate an exponential series shifted to include negative values
y_exp_shifted = np.exp(0.05 * x) + np.random.normal(0, 5, n) - 50

y_yj_transformed, lambda_yj = yeojohnson(y_exp_shifted)

# --- Plotting ---
fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
fig.suptitle('Yeo-Johnson Transformation Example', fontsize=14)

# Original Series subplot
axs[0].plot(x, y_exp_shifted, label="$ y = e^{0.05x} - 50 $", color='tab:blue')
axs[0].set_title("Original Exponential Series (Y-shifted to include negative values)", fontsize=11)
axs[0].set_ylabel("Value")
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].legend(loc='upper left')

# Yeo-Johnson Transformed Series subplot
axs[1].plot(x, y_yj_transformed, color="tab:orange", label=f"$ Y^{{(\\lambda)}} \\text{{ for }} \\lambda={lambda_yj:.2f}$")
axs[1].set_title("Series After Yeo-Johnson Transform", fontsize=11)
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Transformed Value")
axs[1].grid(True, linestyle='--', alpha=0.6)
axs[1].legend(loc='upper left')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

#### Choosing the Right Transformation

Selecting the appropriate transformation depends primarily on the characteristics of your time series data:

- Simple Log Transform ($ ln(Y) $ or $ log_{10}(Y) $):
    - **When to use:** Ideal for data that is strictly positive ($Y > 0$) and where the variance of the series is proportional to its mean, or when there's an exponential growth trend. It's the simplest to interpret.
    - **Limitation:** Cannot handle zero or negative values.
- Box-Cox Transformation:
    - **When to use:** Use when your data is strictly positive ($Y > 0$) and you need to find an optimal power transformation ($\lambda$) to best stabilize variance and/or make the data more normally distributed. It's more flexible than a fixed log transform.
    - **Limitation:** Cannot handle zero or negative values.
- Yeo-Johnson Transformation:
    - **When to use:** This is the most versatile option. Use it when your data contains positive, negative, or zero values. It extends the concept of Box-Cox to accommodate all real numbers, providing an optimal transformation for variance stabilization and normalization across the entire range of data.

In practice, if your data includes non-positive values, Yeo-Johnson is the go-to choice. If your data is strictly positive, both Box-Cox and a simple log transform (if suitable) can be considered, with Box-Cox offering a data-driven optimal parameter.

### Relationship to Normality and Order of Operations

Beyond stabilizing variance for stationarity, these power transformations (especially Box-Cox and Yeo-Johnson) have another significant benefit: they often help make the data more normally (Gaussian) distributed. This is crucial because many statistical models and tests assume or perform better with normally distributed errors. By transforming the data, you can potentially satisfy these assumptions, leading to more reliable model fitting and inference.

It's also important to consider the typical order in which transformations are applied when addressing different types of non-stationarity. Generally, it is recommended to apply variance-stabilising transformations (like log, Box-Cox, or Yeo-Johnson) *before* techniques that address mean non-stationarity, such as differencing or detrending.

This sequence ensures that the variance is stable across the series before attempting to remove trends or seasonality, which can lead to a more effective and interpretable transformation of the time series. For instance, if the variance is increasing with time, differencing a series without first stabilizing its variance might still result in a non-stationary series (in terms of variance) or introduce additional noise.